# Agentic AI

## Goal

An agent is a loop around an LLM and tools.

Fixed chain:
`A → B → C`

Agent:
`Goal → Decide → Act → Observe → Decide again → ... → Answer`

This matches the supplied agent notebook's core idea.

In [ ]:
%pip install -q -U langchain langchain-openai

In [ ]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("OPENAI_API_KEY: ")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

llm=ChatOpenAI(model="gpt-5-mini",temperature=0)

## 1. Create tools

A tool is a function that gives the model an external capability.

The description helps the model choose the right tool.

In [ ]:
@tool
def calculator(expression: str) -> str:
    # Basic educational calculator. Do not expose unrestricted eval to untrusted users.
    allowed=set("0123456789+-*/(). %")
    if not set(expression)<=allowed:
        return "Error: unsupported characters."
    try:
        return str(eval(expression,{"__builtins__":{}},{}))
    except Exception as e:
        return f"Error: {e}"

@tool
def word_count(text: str) -> str:
    return str(len(text.split()))

tools=[calculator,word_count]
tool_map={t.name:t for t in tools}
for t in tools:
    print(t.name,t.description,t.args_schema.model_json_schema())

## 2. Tool calling

Binding tells the model which tools exist.

It does **not** execute them.

In [ ]:
llm_tools=llm.bind_tools(tools)

msg=llm_tools.invoke("Calculate 125 * 8.")
print("text:",msg.content)
print("tool calls:",msg.tool_calls)

In [ ]:
for call in msg.tool_calls:
    result=tool_map[call["name"]].invoke(call["args"])
    print(call["name"],call["args"],"->",result)

## 3. Agent loop

Repeat:
1. ask model
2. if no tool call, stop
3. execute tool
4. return `ToolMessage`
5. ask model again

In [ ]:
def run_agent(query,max_steps=5):
    messages=[HumanMessage(content=query)]

    for step in range(max_steps):
        response=llm_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            tool=tool_map.get(call["name"])
            try:
                result=tool.invoke(call["args"]) if tool else "Unknown tool"
            except Exception as e:
                result=f"Tool error: {e}"

            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            ))

    return "Stopped: maximum steps reached."

print(run_agent("Calculate 17 * 24 and explain the result."))

## 4. RAG as a tool

This is the bridge from RAG to agentic RAG.

Instead of always using RAG, make the retriever one tool among several.

In [ ]:
@tool
def knowledge_search(query: str) -> str:
    # Replace this demo with your Chroma/FAISS retriever.
    if "rag" in query.lower():
        return "RAG means Retrieval-Augmented Generation."
    if "python" in query.lower():
        return "Python is commonly used for data science, ML, automation and AI."
    return "No matching internal document found."

agent_tools=[calculator,word_count,knowledge_search]
print([t.name for t in agent_tools])

## 5. Production safety

Real tools need:
- input validation
- typed schemas
- authentication
- permissions
- rate limits
- timeouts
- retries
- maximum steps
- tracing/logging
- human approval for sensitive actions

An LLM's confidence is not an authorization system.

## 6. Evaluation

Test:
- correct tool selected
- correct arguments
- sensible order
- error recovery
- final answer
- stopping behavior

Use a test set such as:

`request → expected tool(s) → expected behavior`

In [ ]:
tests=[
    ("Calculate 10 * 20.","calculator"),
    ("Count words in: hello world from AI.","word_count"),
]

for query,expected in tests:
    first=llm_tools.invoke(query)
    chosen=first.tool_calls[0]["name"] if first.tool_calls else None
    print("PASS" if chosen==expected else "CHECK",expected,chosen)

## Final mental model

`User goal → LLM decides → Tool → Observation → LLM decides again → ... → Answer`

Agentic AI is not mysterious autonomy. It is a controlled decision/action/observation loop.